In [6]:
!pip3 install pyproj

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import pandas as pd
import rasterio
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

ModuleNotFoundError: No module named 'pyproj'

In [1]:
import xarray as xr
import pandas as pd
import os

import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

/home/tbellagio/.local/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
def load_and_combine_data(file_pattern, folder_path):
    all_data = []  # List to store data from all files

    # Loop through each file in the directory that matches the file pattern
    for filename in os.listdir(folder_path):
        if filename.startswith(file_pattern) and filename.endswith('.nc'):
            # Construct the full file path
            file_path = os.path.join(folder_path, filename)

            # Open the NetCDF file
            ds = xr.open_dataset(file_path)

            # Convert to DataFrame
            df = ds.to_dataframe()

            # Reset index to turn multi-level index into columns
            df.reset_index(inplace=True)

            # Extract latitude and longitude from filename
            parts = filename.replace('.nc', '').split('_')
            lat = parts[-2]
            lon = parts[-1]

            # Add latitude and longitude as constant columns
            df['latitude'] = float(lat)
            df['longitude'] = float(lon)

            # Append DataFrame to list
            all_data.append(df)

            # Close the dataset
            ds.close()

    # Combine all data into a single DataFrame
    combined_df = pd.concat(all_data, ignore_index=True)

    return combined_df

folder_path = '/carnegie/nobackup/scratch/tbellagio/gea_grene-net'

file_pattern = 'era5_monthly_'
combined_data = load_and_combine_data(file_pattern, folder_path)

In [10]:
env_ecotpyes[env_ecotpyes['bio1'].isna()]

,ecotypeid,bio1,bio2,bio3,bio4,bio5,bio6,bio7,bio8,bio9,...,vapr03,vapr04,vapr05,vapr06,vapr07,vapr08,vapr09,vapr10,vapr11,vapr12
22,6184,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
110,9371,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,9394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,9481,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,ecotypeid,longitude,latitude
0,159,3.933330,47.350000
1,265,-1.166670,44.650000
2,763,74.366700,42.300000
3,765,73.400000,42.183300
4,766,73.633300,42.583300
...,...,...,...
226,10011,45.362200,39.869200
227,10013,48.613100,38.740600
228,10014,48.799200,38.653600
229,100001,35.797398,33.093931


In [14]:
combined_data.merge(ecotpyes[['ecotypeid', 'longitude', 'latitude']])

,longitude,latitude,time,t2m,tp,ssrd,ecotypeid
0,18.3175,63.0160,1994-01-01,266.0,266.0,18197108.0,9371
1,18.3175,63.0160,1994-02-01,18197108.0,18197108.0,18197108.0,9371
2,18.3175,63.0160,1994-03-01,18197108.0,18197108.0,18197108.0,9371
3,18.3175,63.0160,1994-04-01,18197108.0,18197108.0,18197108.0,9371
4,18.3175,63.0160,1994-05-01,18197108.0,18197108.0,18197108.0,9371
...,...,...,...,...,...,...,...
715,18.4522,62.8892,2023-08-01,18564596.0,18564596.0,18564596.0,6184
716,18.4522,62.8892,2023-09-01,18564596.0,18564596.0,18564596.0,6184
717,18.4522,62.8892,2023-10-01,18564596.0,18564596.0,18564596.0,6184
718,18.4522,62.8892,2023-11-01,18564596.0,18564596.0,18564596.0,6184


In [5]:
path_metadata = '/home/tbellagio/grene/data/'
env_ecotpyes = pd.read_csv(path_metadata + 'worldclim_ecotypesdata.csv')

In [6]:
ecotpyes = pd.read_csv(path_metadata + 'ecotypes_data.csv')

In [7]:
ecotpyes.to_csv('ecotypes_data.csv')

In [ ]:
def extract_values(file_path, points_df):
    """
    Extracts raster values at specified points.
    
    Args:
    file_path (str): Path to the GeoTIFF file.
    points_df (pd.DataFrame): DataFrame containing 'latitude' and 'longitude' columns.
    
    Returns:
    pd.Series: Extracted values.
    """
    # Convert DataFrame to GeoDataFrame
    gdf = gpd.GeoDataFrame(
        points_df, 
        geometry=gpd.points_from_xy(points_df.longitude, points_df.latitude),
        crs="EPSG:4326"
    )

    # Open the raster file
    with rasterio.open(file_path) as src:
        # Reproject points to the raster's CRS
        gdf = gdf.to_crs(src.crs)
        
        # Extract the raster values at the points
        values = [val[0] for val in src.sample([(pt.x, pt.y) for pt in gdf.geometry])]
        
    return pd.Series(values)

# Example usage
file_path = 'path_to_your_geotiff_file.tif'
points = pd.DataFrame({
    'latitude': [10.0, 20.0, -10.0],
    'longitude': [45.0, 85.0, -75.0]
})

values = extract_values(file_path, points)
print(values)
